In [ ]:
%pip install Pillow
%pip install --upgrade google-genai

In [ ]:
import google.auth
_, PROJECT_ID = google.auth.default()

In [ ]:
from google.genai import types
from google import genai
import concurrent.futures
from PIL import Image
from collections.abc import Callable

def process_image_embedding_worker(id: str, file_path: str):
    client = genai.Client(vertexai=True, location="global", project=PROJECT_ID)
    try:            
        # 임베딩 API 호출
        result = client.models.embed_content(
            model='gemini-embedding-2',
            contents=[
                types.Part.from_uri(file_uri=file_path),
            ],
            config = types.EmbedContentConfig(
                output_dimensionality=768,
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(
                        attempts=100,
                        initial_delay=1.0,
                        max_delay=5.0
                    )
                )
            )
        )        
        return [id, file_path, result.embeddings[0].values]
    except Exception as e:
        print(f"API 호출 또는 처리 중 오류 발생 ({id}): {e}")
        return None

def process_text_content_embedding_worker(id: str, content: str):
    client = genai.Client(vertexai=True, location="global", project=PROJECT_ID)
    try:            
        # 임베딩 API 호출
        result = client.models.embed_content(
            model='gemini-embedding-2',
            contents=[
                types.Part.from_text(text=f"title: none | text: {content}"),
            ],
            config = types.EmbedContentConfig(
                output_dimensionality=768,
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(
                        attempts=100,
                        initial_delay=1.0,
                        max_delay=5.0
                    )
                )
            )
        )        
        return [id, content, result.embeddings[0].values]
    except Exception as e:
        print(f"API 호출 또는 처리 중 오류 발생 ({id}): {e}")
        return None

def process_image_thumbnail_worker(id: str, file_path: str):
    output = f"./thumbnails/{id}.webp"
    with Image.open(file_path) as img:
        # RGB 모드로 변환 (PNG나 특정 JPG의 투명도/색상 프로필 오류 방지)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")        
        # 한 변의 최대 해상도를 400px로 제한 (비율 유지)
        img.thumbnail((400, 400))        
        # WebP 포맷으로 저장 (quality는 0~100 사이, 80 정도면 화질 저하 없이 용량이 대폭 줄어듭니다)
        img.save(output, format="WEBP", quality=80)
    return [id, file_path, output]

def process_parallel(tasks: list, processor: Callable[[str], str], max_workers: int = 4):
    """
    여러개의 Task를 ThreadPoolExecutor를 이용해 병렬로 동작합니다.
    """
    results_with_index = []
    
    print(f"총 {len(tasks)}개의 데이터를 {max_workers}개의 스레드로 처리합니다...")
    
    # 스레드 풀을 이용한 병렬 처리
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 입력된 Task 리스트의 인덱스를 유지하여 순서 보장
        future_to_index = {
            executor.submit(processor, id, data): idx
            for idx, (id, data) in enumerate(tasks)
        }
        
        # 완료되는 순서대로 결과 수집
        i = 0
        for future in concurrent.futures.as_completed(future_to_index):
            i += 1
            if i % 1000 == 0:
                print(f"{i} 개 처리 완료")
            idx = future_to_index[future]
            try:
                res = future.result()
                if res:
                    results_with_index.append((idx, res))
            except Exception as e:
                print(f"스레드 실행 중 예외 발생 (Index {idx}): {e}")

    # 병렬 처리로 인해 뒤섞인 결과를 원래 입력 순서(index)대로 정렬
    results_with_index.sort(key=lambda x: x[0])
    
    # 최종 결과 데이터만 리스트로 반환
    final_results = [res[1] for res in results_with_index]
    return final_results

In [ ]:
# ! 명령어를 통해 gcloud 결과를 file_list 변수에 리스트 형태로 저장
# ** 와일드카드를 사용해 하위 폴더의 모든 파일을 재귀적으로 탐색합니다.
file_list = !gcloud storage ls gs://jk-amazon-products/**

print(f"총 {len(file_list)}개의 파일을 찾았습니다.")
images_list = []

for file in file_list:
    if file.endswith(('.png', '.jpg')):
        # Convert gs:// to public gcs path
        images_list.append("https://storage.googleapis.com/" + file[5:])
print(f"총 {len(images_list)}개의 이미지 파일을 찾았습니다.")

In [ ]:
embeddings = process_images_parallel(images_list, process_image_embedding_worker, 64)

In [ ]:
import json
# JSONL 파일로 저장하기
output_file_path = "images.jsonl"

# 'w' 모드로 파일을 열고, 한 줄씩 json.dumps()를 이용해 문자열로 변환 후 줄바꿈(\n)을 더해줍니다.
with open(output_file_path, "w", encoding="utf-8") as f:
    for item in embeddings:
        # ensure_ascii=False를 주면 한글이나 특수문자가 유니코드 형태로 깨지는 것을 방지합니다.
        json_line = json.dumps({"id": item[0], "vectors": {"image_embedding": item[2]}}, ensure_ascii=False)
        f.write(json_line + "\n")

In [2]:
!gcloud storage cp -r amazon_products.jsonl gs://jk-amazon-products-index/data/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://amazon_products.jsonl to gs://jk-amazon-products-index/data/amazon_products.jsonl
  Completed files 32/1 | 3.9GiB/3.9GiB | 1.1GiB/s                              

Average throughput: 784.4MiB/s


In [ ]:
# Run this at terminal
PROJECT_ID="sandbox-373102"
LOCATION="asia-northeast1"
COLLECTION_ID="amazon-product-dataset-768"

# 2. curl로 REST API 호출
curl -X POST \
  "https://vectorsearch.googleapis.com/v1beta/projects/${PROJECT_ID}/locations/${LOCATION}/collections?collection_id=${COLLECTION_ID}" \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -d '{
    "vector_schema": {
      "image_embedding": {
        "dense_vector": {
          "dimensions": 768
        }
      }
    }
  }'

In [1]:
! gcloud vector-search collections import-data-objects amazon-product-dataset-768 --gcs-import-contents-uri="gs://jk-amazon-products-index/data/" --gcs-import-error-uri="gs://jk-amazon-products-index/error/" --location="asia-northeast1"

ERROR: (gcloud) Invalid choice: 'vector-search'.
Maybe you meant:
  gcloud sql import bak
  gcloud sql import csv
  gcloud sql import sql
  gcloud sql import tde
  gcloud composer environments storage data import
  gcloud datastore import
  gcloud firestore import
  gcloud healthcare dicom-stores import gcs
  gcloud healthcare fhir-stores import gcs
  gcloud healthcare hl7v2-stores import gcs

To search the help text of gcloud commands, run:
  gcloud help -- SEARCH_TERMS


In [ ]:
gcloud beta vector-search collections indexes create idx-amazon-product-dataset-768 \
  --collection=amazon-product-dataset-768 \
  --index-field=image_embedding \
  --location="asia-northeast1"